<a href="https://colab.research.google.com/github/mfernandezzz/Pruebas_Tecnicas/blob/main/TT_stackoverflow_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv('/content/results.csv', low_memory=False)
schema = pd.read_csv('/content/schema.csv')
pd.set_option('display.max_columns', None)

In [3]:
df.shape

(49191, 172)

In [4]:
df.info(memory_usage='deep')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 49191 entries, 0 to 49190
Columns: 172 entries, ResponseId to JobSat
dtypes: float64(52), int64(1), object(119)
memory usage: 357.8 MB


In [5]:
for c in df.select_dtypes('object'):
  df[c] = df[c].astype('category')

In [6]:
for i in df.select_dtypes('integer'):
  df[i] = pd.to_numeric(df[i], downcast='integer')

In [7]:
for f in df.select_dtypes('float'):
  df[f] = pd.to_numeric(df[f], downcast='float')

In [8]:
df.info(memory_usage='deep')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 49191 entries, 0 to 49190
Columns: 172 entries, ResponseId to JobSat
dtypes: category(119), float32(50), float64(2), int32(1)
memory usage: 57.6 MB


In [9]:
df.columns = (
    df.columns
    .str.lower()
    .str.strip()
    .str.replace(' ', '_')
)
df.columns

Index(['responseid', 'mainbranch', 'age', 'edlevel', 'employment',
       'employmentaddl', 'workexp', 'learncodechoose', 'learncode',
       'learncodeai',
       ...
       'aiagentorchestration', 'aiagentorchwrite', 'aiagentobservesecure',
       'aiagentobswrite', 'aiagentexternal', 'aiagentextwrite', 'aihuman',
       'aiopen', 'convertedcompyearly', 'jobsat'],
      dtype='object', length=172)

In [10]:
nulls_count = df.isnull().sum()
nulls_count[nulls_count > 0].sort_values(ascending=False)

,0
aiagentobswrite,48927
sotagswant_entry,48761
sotagshaveentry,48733
aimodelswantentry,48716
aiagentorchwrite,48713
...,...
employmentaddl,4316
learncodeai,3990
learncodechoose,2333
edlevel,1042


In [11]:
nulls_percentage = df.isnull().mean()*100
nulls_percentage[nulls_percentage > 0].sort_values(ascending=False)

,0
aiagentobswrite,99.463316
sotagswant_entry,99.125856
sotagshaveentry,99.068935
aimodelswantentry,99.034376
aiagentorchwrite,99.028278
...,...
employmentaddl,8.773963
learncodeai,8.111240
learncodechoose,4.742737
edlevel,2.118274


In [12]:
duplicates_sum = df.duplicated().sum()
duplicates_sum

np.int64(0)

In [13]:
# duplicates_percentage = (df.duplicated().sum()/df.shape[0]) * 100
# duplicates_percentage

In [14]:
# show_duplicates = df[df.duplicated(keep=False)].sort_values(by=list(df.columns)).head()
# show_duplicates

In [ ]:
# df = df.drop_duplicates()

**What percentage of people from each country knows Python?**

In [80]:
schema.columns

Index(['qid', 'qname', 'question', 'force_resp', 'type', 'selector'], dtype='object')

In [81]:
schema['qname'].unique()

array(['MainBranch', 'Age', 'Employment', 'RemoteWork', 'Check',
       'CodingActivities', 'EdLevel', 'LearnCode', 'LearnCodeOnline',
       'TechDoc', 'YearsCode', 'YearsCodePro', 'DevType', 'OrgSize',
       'PurchaseInfluence', 'BuyNewTool', 'BuildvsBuy', 'TechEndorse',
       'Country', 'Currency', 'CompTotal', 'Language', 'Database',
       'Platform', 'Webframe', 'Embedded', 'MiscTech', 'ToolsTech',
       'NEWCollabTools', 'OpSys', 'OfficeStackAsync', 'OfficeStackSync',
       'AISearchDev', 'NEWSOSites', 'SOVisitFreq', 'SOAccount',
       'SOPartFreq', 'SOHow', 'SOComm', 'AISelect', 'AISent', 'AIBen',
       'AIAcc', 'AIComplex', 'AITool', 'AINext', 'AIThreat', 'AIEthics',
       'AIChallenges', 'TBranch', 'ICorPM', 'WorkExp', 'Knowledge',
       'Frequency', 'TimeSearching', 'TimeAnswering', 'Frustration',
       'ProfessionalTech', 'ProfessionalCloud', 'ProfessionalQuestion',
       'Industry', 'JobSat', 'JobSatPoints', 'SOTeamsUsage',
       'SurveyLength', 'SurveyEase', 'K

Groups of countries

In [92]:
country_grp = df.groupby('country', observed=False)

Countries and the respective amount of respondents

In [93]:
country_resp = pd.DataFrame(df['country'].value_counts())
country_resp.rename(columns={'count': 'respondents'}, inplace=True)
country_resp.head(10)

,respondents
country,
United States of America,7233
Germany,3025
India,2547
United Kingdom of Great Britain and Northern Ireland,2042
France,1409
Canada,1305
Ukraine,964
Poland,888
Netherlands,867


Countries and the respective amount of python users

In [94]:
country_py = country_grp['languagehaveworkedwith'].apply(lambda x: x.str.contains('Python').sum()).sort_values(ascending=False)
country_python_data = pd.DataFrame(country_py)
country_python_data.head(10)

,languagehaveworkedwith
country,
United States of America,4221
Germany,1685
India,1162
United Kingdom of Great Britain and Northern Ireland,1027
France,820
Canada,687
Poland,459
Netherlands,458
Italy,436


Country, number of respondents, python users and the respective percentage of Python users by country.

In [95]:
python_df = pd.concat([country_resp, country_py], axis='columns').reset_index() # sort_values(by='responseid', ascending=False)
python_df.rename(columns={'languagehaveworkedwith': 'py_users'}, inplace=True)
python_df['%'] = round((python_df['py_users'] / python_df['respondents']) * 100, 2)
python_df.head(10)

,country,respondents,py_users,%
0,United States of America,7233,4221,58.36
1,Germany,3025,1685,55.70
2,India,2547,1162,45.62
3,United Kingdom of Great Britain and Northern I...,2042,1027,50.29
4,France,1409,820,58.20
5,Canada,1305,687,52.64
6,Ukraine,964,384,39.83
7,Poland,888,459,51.69
8,Netherlands,867,458,52.83
9,Italy,835,436,52.22


Other solution

In [96]:
paises_individuos = pd.DataFrame(df['country'].value_counts())
paises_individuos.rename(columns={'count': 'individuos'}, inplace=True)
paises_individuos.head()

,individuos
country,
United States of America,7233
Germany,3025
India,2547
United Kingdom of Great Britain and Northern Ireland,2042
France,1409


In [97]:
filt_python = (df['languagehaveworkedwith'].str.contains('Python')) & (df['languagehaveworkedwith'].notna())
paises_usuarios_python = df.loc[filt_python]
paises_usuarios_python_data = pd.DataFrame(paises_usuarios_python['country'].value_counts())
paises_usuarios_python_data.rename(columns={'count': 'py_users'}, inplace=True)
paises_usuarios_python_data.head(10)

,py_users
country,
United States of America,4221
Germany,1685
India,1162
United Kingdom of Great Britain and Northern Ireland,1027
France,820
Canada,687
Poland,459
Netherlands,458
Italy,436


In [98]:
data_python = pd.concat([paises_individuos, paises_usuarios_python_data], axis='columns').reset_index()
data_python['%'] = round((data_python['py_users'] / data_python['individuos']) * 100, 2)
data_python.head(10)

,country,individuos,py_users,%
0,United States of America,7233,4221,58.36
1,Germany,3025,1685,55.70
2,India,2547,1162,45.62
3,United Kingdom of Great Britain and Northern I...,2042,1027,50.29
4,France,1409,820,58.20
5,Canada,1305,687,52.64
6,Ukraine,964,384,39.83
7,Poland,888,459,51.69
8,Netherlands,867,458,52.83
9,Italy,835,436,52.22
